# Overrepresented techniques in the ransomed websites

In [4]:
# set the working directory
setwd("/Users/dmk6603/Documents/ransom_victims/10-r_scripts")
# hide warnings
options(warn=-1)

In [12]:
# =============================================================================
# Odds Ratio Analysis: Overrepresented Technologies in Ransomed Sites
# Uses epitools to compute OR for each technology
# =============================================================================

library(tidyverse)
library(jsonlite)
library(epitools)

# -----------------------------------------------------------------------------
# 1. Load data
# -----------------------------------------------------------------------------

# Global product installs (baseline population)
global_installs <- read_csv("/Users/dmk6603/Documents/ransom_victims/9-paper/analysis_scripts/data/202303_Tech_Install_Universe_Counts.csv", show_col_types = FALSE) %>%
  mutate(
    total_enterprises = as.numeric(gsub(",", "", `Total Enterprises`))
  ) %>%
  select(product_name = Product, total_enterprises) %>%
  distinct(product_name, .keep_all = TRUE)

# Ransomed sites
ransomed_sites <- read_csv("/Users/dmk6603/Documents/ransom_victims/6-main/data/data_tech_swdb.csv", show_col_types = FALSE)

# -----------------------------------------------------------------------------
# 2. Parse technologies JSON from ransomed sites
# -----------------------------------------------------------------------------

ransomed_technologies <- ransomed_sites %>%
  rowwise() %>%
  mutate(tech_list = list(fromJSON(technologies))) %>%
  ungroup() %>%
  unnest(tech_list) %>%
  select(site_url, product_name = product, vendor, category)

# Count how many ransomed sites use each product
ransomed_product_counts <- ransomed_technologies %>%
  distinct(site_url, product_name) %>%
  count(product_name, name = "ransomed_with_tech")

# Keep vendor and category info per product (take the most common if duplicates)
product_info <- ransomed_technologies %>%
  count(product_name, vendor, category, sort = TRUE) %>%
  distinct(product_name, .keep_all = TRUE) %>%
  select(product_name, vendor, category)

total_ransomed_sites <- n_distinct(ransomed_sites$site_url)

# -----------------------------------------------------------------------------
# 3. Define total population
# -----------------------------------------------------------------------------

# Use the max enterprise count as a proxy for the total universe.
# Replace this with a better number if you have one.
total_population <- 5620161

cat("Total ransomed sites:", total_ransomed_sites, "\n")
cat("Estimated total enterprise population:", total_population, "\n\n")

# -----------------------------------------------------------------------------
# 4. Build 2x2 tables and compute odds ratios
# -----------------------------------------------------------------------------

#                    Ransomed    Not Ransomed
# Has tech      (a)              (b)
# No tech       (c)              (d)

analysis_data <- ransomed_product_counts %>%
  inner_join(global_installs, by = "product_name") %>%
  mutate(
    a = ransomed_with_tech,
    b = total_enterprises - ransomed_with_tech,
    c = total_ransomed_sites - ransomed_with_tech,
    d = (total_population - total_enterprises) - c
  ) %>%
  filter(a > 0, b > 0, c > 0, d > 0)

compute_odds_ratio <- function(a, b, c, d) {
  contingency_table <- matrix(c(a, c, b, d), nrow = 2, byrow = FALSE)
  result <- oddsratio.wald(contingency_table)
  tibble(
    odds_ratio = result$measure[2, "estimate"],
    ci_lower   = result$measure[2, "lower"],
    ci_upper   = result$measure[2, "upper"],
    p_value    = result$p.value[2, "fisher.exact"]
  )
}

results <- analysis_data %>%
  rowwise() %>%
  mutate(or_result = list(compute_odds_ratio(a, b, c, d))) %>%
  unnest(or_result) %>%
  ungroup() %>%
  arrange(desc(odds_ratio))

# -----------------------------------------------------------------------------
# 5. Adjust for multiple comparisons (Benjamini-Hochberg)
# -----------------------------------------------------------------------------

results <- results %>%
  mutate(p_adjusted = p.adjust(p_value, method = "BH")) %>%
  left_join(product_info, by = "product_name") %>%
  select(
    vendor, product_name, category,
    ransomed_with_tech, total_enterprises,
    a, b, c, d,
    odds_ratio, ci_lower, ci_upper, p_value, p_adjusted
  )

# -----------------------------------------------------------------------------
# 6. Output
# -----------------------------------------------------------------------------

cat("=== Top 20 Overrepresented Technologies (by Odds Ratio) ===\n\n")
print(head(results, 20), n = 20, width = Inf)

cat("\n=== Statistically Significant (adjusted p < 0.05) ===\n\n")
significant <- results %>% filter(p_adjusted < 0.05)
print(significant, n = 50, width = Inf)

write_csv(results, "odds_ratio_results.csv")
cat("\nFull results saved to odds_ratio_results.csv\n")

Total ransomed sites: 4816 
Estimated total enterprise population: 5620161 

=== Top 20 Overrepresented Technologies (by Odds Ratio) ===

# A tibble: 20 × 14
   vendor                     product_name                     
   <chr>                      <chr>                            
 1 GOG                        Galaxy Client                    
 2 ISGN                       MORvision                        
 3 Parallelus                 Parallelus                       
 4 NETQ Healthcare            NetQ                             
 5 Brave Bison Group Plc      Brave Bison                      
 6 SearchFit                  SearchFit                        
 7 VUE Software               VUE Billing and Collections      
 8 Hirewire Inc               HireWire                         
 9 Cactusoft                  Kartris                          
10 logicsource                Procurement Software             
11 SmarterTools               SmarterStats                     
12 Goldsta